# 14 — Organization-Wide Skill Gap
Roll up company-wide skill gaps. Severity thresholds scaled to 500-person population.

In [1]:

import pandas as pd
import numpy as np
import os

PROC = r'../data/processed'

skill_gap_df = pd.read_csv(f'{PROC}/skill_gap_detailed.csv')
emp_gap_summary = pd.read_csv(f'{PROC}/employee_skill_gap_summary.csv')
print(f"Skill gap detailed: {skill_gap_df.shape}")
print(f"Employee gap summary: {emp_gap_summary.shape}")
n_employees = emp_gap_summary['employee_id'].nunique()
print(f"Employees in analysis: {n_employees}")


Skill gap detailed: (5000, 7)
Employee gap summary: (500, 7)
Employees in analysis: 500


In [2]:

# ── Company-wide skill gap rollup ──
# For each skill: how many employees lack it (absolute count) and at what weighted importance
org_skill_gap = skill_gap_df[skill_gap_df['is_gap']].groupby('skill').agg(
    employees_lacking=('employee_id', 'nunique'),
    avg_importance=('importance', 'mean'),
    total_gap_weight=('importance', 'sum')
).reset_index().sort_values('total_gap_weight', ascending=False)

org_skill_gap['pct_employees_lacking'] = (org_skill_gap['employees_lacking'] / n_employees * 100).round(1)
org_skill_gap['avg_importance'] = org_skill_gap['avg_importance'].round(3)
org_skill_gap['total_gap_weight'] = org_skill_gap['total_gap_weight'].round(2)

print(f"Distinct skills with gaps: {len(org_skill_gap)}")
print(f"\nTop 20 org-wide skill gaps (by weighted importance):")
print(org_skill_gap.head(20).to_string(index=False))


Distinct skills with gaps: 10

Top 20 org-wide skill gaps (by weighted importance):
                skill  employees_lacking  avg_importance  total_gap_weight  pct_employees_lacking
              Writing                324           3.916           1268.74                   64.8
Reading Comprehension                309           4.050           1251.37                   61.8
    Critical Thinking                300           4.036           1210.66                   60.0
           Monitoring                312           3.738           1166.27                   62.4
     Active Listening                286           4.017           1148.88                   57.2
             Speaking                290           3.894           1129.23                   58.0
      Active Learning                299           3.346           1000.50                   59.8
          Mathematics                308           3.017            929.24                   61.6
  Learning Strategies             

In [3]:

# ── Severity thresholds ──
# Scaled to 500-person population:
#   HIGH:   >30% of employees (>150 people) lack the skill AND avg_importance >= 3.5
#   MEDIUM: >15% (>75 people) lack the skill AND avg_importance >= 3.0
#   LOW:    anything else
# Rationale:
#   - 30%: if nearly 1 in 3 employees can't perform a critical skill, it's a systemic gap
#   - 15%: 1 in 6 is material for planning purposes but not immediately critical
#   - Importance >= 3.5/5: O*NET defines skills rated >= 3.5 as "important to very important"

HIGH_PCT = 30.0
HIGH_IMP = 3.5
MED_PCT = 15.0
MED_IMP = 3.0

def assign_severity(row):
    if row['pct_employees_lacking'] >= HIGH_PCT and row['avg_importance'] >= HIGH_IMP:
        return 'HIGH'
    elif row['pct_employees_lacking'] >= MED_PCT and row['avg_importance'] >= MED_IMP:
        return 'MEDIUM'
    else:
        return 'LOW'

org_skill_gap['severity'] = org_skill_gap.apply(assign_severity, axis=1)

severity_counts = org_skill_gap['severity'].value_counts()
print("=== Severity Thresholds ===")
print(f"HIGH   (>={HIGH_PCT}% employees lacking AND importance>={HIGH_IMP}): {severity_counts.get('HIGH',0)} skills")
print(f"MEDIUM (>={MED_PCT}% employees lacking AND importance>={MED_IMP}): {severity_counts.get('MEDIUM',0)} skills")
print(f"LOW    (everything else): {severity_counts.get('LOW',0)} skills")
print(f"\nThreshold rationale:")
print(f"  {HIGH_PCT}% = 1 in 3 of 500 employees — systemic gap requiring immediate L&D investment")
print(f"  {MED_PCT}% = 1 in 6 of 500 employees — material planning-level gap")
print(f"  Importance >= {HIGH_IMP}: O*NET 'important to very important' range on 1-5 scale")


=== Severity Thresholds ===
HIGH   (>=30.0% employees lacking AND importance>=3.5): 6 skills
MEDIUM (>=15.0% employees lacking AND importance>=3.0): 3 skills
LOW    (everything else): 1 skills

Threshold rationale:
  30.0% = 1 in 3 of 500 employees — systemic gap requiring immediate L&D investment
  15.0% = 1 in 6 of 500 employees — material planning-level gap
  Importance >= 3.5: O*NET 'important to very important' range on 1-5 scale


In [4]:

# ── Department-level skill gap ──
# skill_gap_df already has 'role' col — drop it to avoid role_x/role_y conflict
sgd_no_role = skill_gap_df[skill_gap_df['is_gap']].drop(columns=['role'], errors='ignore')
dept_gap = sgd_no_role.merge(
    emp_gap_summary[['employee_id','role']], on='employee_id'
).groupby('role').agg(
    total_gaps=('is_gap', 'sum'),
    unique_employees=('employee_id', 'nunique'),
    avg_gap_importance=('importance', 'mean')
).reset_index().sort_values('avg_gap_importance', ascending=False)

print("\n=== Skill Gaps by Role ===")
print(dept_gap.to_string(index=False))



=== Skill Gaps by Role ===
            role  total_gaps  unique_employees  avg_gap_importance
      Hr Manager         230                38            3.699913
    Hr Executive         242                39            3.571322
        Helpdesk         248                42            3.570000
    Content Lead         221                38            3.563213
 Account Manager         301                50            3.553854
        Engineer         150                25            3.544600
 Sales Executive         301                49            3.539701
      Accountant         236                39            3.536695
Support Engineer         240                40            3.525667
     Seo Analyst         207                34            3.475362
          Tester         183                30            3.469781
         Auditor         237                38            3.162236
       Developer         225                38            3.094400


In [5]:

# ── Save ──
org_skill_gap.to_csv(f'{PROC}/org_skill_gap.csv', index=False)
dept_gap.to_csv(f'{PROC}/role_skill_gap.csv', index=False)
print("Saved: org_skill_gap.csv, role_skill_gap.csv")
print(f"\nHIGH severity skills:\n{org_skill_gap[org_skill_gap['severity']=='HIGH']['skill'].values[:10]}")


Saved: org_skill_gap.csv, role_skill_gap.csv

HIGH severity skills:
['Writing' 'Reading Comprehension' 'Critical Thinking' 'Monitoring'
 'Active Listening' 'Speaking']


**Org-wide skill gap complete.** Severity thresholds: HIGH (≥30% employees, importance≥3.5), MEDIUM (≥15%, importance≥3.0), LOW (everything else). Scaled to 500-person population.